In [1]:
import sys
from google import genai
sys.path.append('../')
from src.load_data import *
from src.agents import *
from src.fine_tuned_agents import *

In [2]:
testing_data = load_data_from_csv('../data/test_sent_emo.csv')
print(testing_data.head())

Data loaded successfully from ../data/test_sent_emo.csv
   Sr No.                                          Utterance Speaker  \
0       1  Why do all youre coffee mugs have numbers on ...    Mark   
1       2  Oh. Thats so Monica can keep track. That way ...  Rachel   
2       3                                       Y'know what?  Rachel   
3      19                     Come on, Lydia, you can do it.    Joey   
4      20                                              Push!    Joey   

    Emotion Sentiment  Dialogue_ID  Utterance_ID  Season  Episode  \
0  surprise  positive            0             0       3       19   
1     anger  negative            0             1       3       19   
2   neutral   neutral            0             2       3       19   
3   neutral   neutral            1             0       1       23   
4       joy  positive            1             1       1       23   

      StartTime       EndTime  
0  00:14:38,127  00:14:40,378  
1  00:14:40,629  00:14:47,385  


In [3]:
testing_data.drop_duplicates(inplace=True)

In [4]:
# Create a unique key like "102_5" (Dialogue_ID + Utterance_ID)
testing_data['Recognition_ID'] = (
    testing_data['Dialogue_ID'].astype(str) + "_" + 
    testing_data['Utterance_ID'].astype(str) + "_" +
    testing_data['Season'].astype(str) + "_" +
    testing_data['Episode'].astype(str)
)

print(testing_data[['Dialogue_ID', 'Utterance_ID', 'Recognition_ID']].head())

   Dialogue_ID  Utterance_ID Recognition_ID
0            0             0       0_0_3_19
1            0             1       0_1_3_19
2            0             2       0_2_3_19
3            1             0       1_0_1_23
4            1             1       1_1_1_23


In [8]:
def preprocess_test_data(testing_data):
    df = testing_data.copy()
    # Ensure chronological order
    df = df.sort_values(['Dialogue_ID', 'Utterance_ID'])
    
    scenes = []
    for diag_id, group in df.groupby('Dialogue_ID'):
        # 1. The 'Public' data we give to agents (No labels!)
        agent_view = group[['Utterance', 'Speaker', 'Recognition_ID']].to_dict(orient='records')
        
        # 2. The 'Private' data for F1 calculation
        ground_truth = group[['Recognition_ID', 'Emotion', 'Sentiment']].to_dict(orient='records')
        
        scene_data = {
            "dialogue_id": diag_id,
            "utterances": agent_view,  # Sent to Agents
            "ground_truth": ground_truth, # Saved for Evaluation
            "total_turns": len(group)
        }
        scenes.append(scene_data)
    return scenes

preprocessed_test_data = preprocess_test_data(testing_data)
preprocessed_test_data_df = pd.DataFrame(preprocessed_test_data)
print(preprocessed_test_data_df.head())
print(preprocessed_test_data[0])
# Check exactly what the agents are seeing
print(f"Available keys in your data: {preprocessed_test_data[0]['utterances'][0].keys()}")

   dialogue_id                                         utterances  \
0            0  [{'Utterance': 'Why do all youre coffee mugs ...   
1            1  [{'Utterance': 'Come on, Lydia, you can do it....   
2            2  [{'Utterance': 'Okay.', 'Speaker': 'Ross', 'Re...   
3            3  [{'Utterance': 'Oh, okay, I get it.', 'Speaker...   
4            4  [{'Utterance': 'Ohh!', 'Speaker': 'Phoebe', 'R...   

                                        ground_truth  total_turns  
0  [{'Recognition_ID': '0_0_3_19', 'Emotion': 'su...            3  
1  [{'Recognition_ID': '1_0_1_23', 'Emotion': 'ne...            8  
2  [{'Recognition_ID': '2_0_5_16', 'Emotion': 'ne...           11  
3  [{'Recognition_ID': '3_0_5_15', 'Emotion': 'ne...            7  
4  [{'Recognition_ID': '4_0_4_14', 'Emotion': 'su...            9  
{'dialogue_id': 0, 'utterances': [{'Utterance': 'Why do all you\x92re coffee mugs have numbers on the bottom?', 'Speaker': 'Mark', 'Recognition_ID': '0_0_3_19'}, {'Utterance': '

In [9]:
def run_phase2_council(scene_obj):
    """
    Processes a single scene using the keys present in your data.
    """
    dialogue_id = scene_obj["dialogue_id"]
    utterances = scene_obj["utterances"]
    
    # Format script for Level 1 agents
    scene_script = "\n".join([f"{u['Speaker']}: {u['Utterance']}" for u in utterances])

    # -- LEVEL 1: GLOBAL CONTEXT --
    global_context = groq_llm_call(
        prompt=f"{context_manager_prompt}\n\nScene:\n{scene_script}", 
        model="meta-llama/llama-4-scout-17b-16e-instruct", 
        api_key=context_manager_api
    )
    social_graph = groq_llm_call(
        prompt=f"{relational_graph_prompt}\n\nScene:\n{scene_script}", 
        model="meta-llama/llama-4-maverick-17b-128e-instruct", 
        api_key=relational_graph_api
    )

    scene_results = []

    # -- LEVEL 2 & 3: UTTERANCE PROCESSING --
    for u in utterances:
        # Use .get() to avoid KeyErrors if a row is malformed
        target_text = u.get('Utterance', "")
        speaker = u.get('Speaker', "Unknown")
        rec_id = u.get('Recognition_ID', "unknown_id") # Critical for F1 mapping

        with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
            f3 = executor.submit(call_tuned_profiler, target_text, social_graph)
            f4 = executor.submit(call_tuned_sentiment, target_text)
            
            profile_report = f3.result()
            sentiment_report = f4.result()

        dynamics_report = call_social_dynamics(target_text, profile_report, social_graph)

        # Agent 6: Final Aggregator
        # We pass rec_id so it can be preserved in the JSON output
        final_prediction_raw = call_gpt_oss_aggregator(
            rec_id, target_text, global_context, profile_report, sentiment_report, dynamics_report
        )

        scene_results.append({
            "Dialogue_ID": dialogue_id,
            "Recognition_ID": rec_id,
            "Speaker": speaker,
            "Utterance": target_text,
            "Predicted_Emotion_Raw": final_prediction_raw,
            "Actual_Emotion": u.get('Emotion', 'neutral') # Ground truth check
        })
    
    return scene_results

In [11]:
import pandas as pd
import time

all_final_results = []
output_file = "../logs/council_phase2_results.jsonl"

# Loop through your preprocessed scenes
for i, scene in enumerate(preprocessed_test_data):
    print(f"🎬 Processing Scene {i+1}/{len(preprocessed_test_data)} (ID: {scene['dialogue_id']})...")
    
    try:
        scene_out = run_phase2_council(scene)
        all_final_results.extend(scene_out)
        
        # Every 5 scenes, save progress to disk
        if (i + 1) % 5 == 0:
            pd.DataFrame(all_final_results).to_csv(output_file, index=False)
            print(f"💾 Progress saved to {output_file}")
            print(all_final_results[-1]) # Print last result for sanity check
            
    except Exception as e:
        print(f"⚠️ Error in Scene {scene['dialogue_id']}: {e}")
        break # break to next scene so the whole run doesn't die
    if i == 5:
        break # Remove this after testing the first scene end-to-end
        

# Final Save
pd.DataFrame(all_final_results).to_csv(output_file, index=False)
print("✅ ALL SCENES PROCESSED SUCCESSFULLY!")

🎬 Processing Scene 1/280 (ID: 0)...
🎬 Processing Scene 2/280 (ID: 1)...
🎬 Processing Scene 3/280 (ID: 2)...
🎬 Processing Scene 4/280 (ID: 3)...
⚠️ Error in Scene 3: 429 Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.
✅ ALL SCENES PROCESSED SUCCESSFULLY!
